# Train Transolver on the SU2 hypersonic dataset

This notebook drives the training runs on Kaggle. Attach the
`zeteixeira/su2-hypersonic-sphere-cone` dataset as input, set the accelerator
to T4 x2 in the notebook editor, turn the internet on, and run every cell.

A push from the command line cannot choose the accelerator and usually draws a
P100, which the Kaggle build of torch does not support. `scripts/train.py`
stops within seconds on that card. Use **Copy & Edit** on a notebook already
set to T4 x2: the copy keeps the accelerator.

The notebook runs one of two things. The ensemble cell trains one member of the
five-member deep ensemble at `M = 32`. Every member shares the split seed and
differs only in `--init-seed`, from 0 to 4. The prior-ablation cells train the
low-data cells instead.

Run one seed per copy of the notebook. Kaggle keeps only the output of the
latest version, so a second push to the same notebook deletes the checkpoints
of the first. Output lands in `/kaggle/working/run_*/`. One 350-epoch member
takes about 1.3 hours.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile

REPO = "/kaggle/working/transolver-hypersonic"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/zeteixeira03/transolver-hypersonic.git", REPO],
                   check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "einops", "-e", REPO],
               check=True)

MIN_CASES = 750

# Kaggle may extract an uploaded archive into a subdirectory, so each file is found on its own
INPUT = "/kaggle/input"
assert os.path.isdir(INPUT), "no /kaggle/input at all: attach the dataset"
ledger_path, zip_path, case_dir, case_n = None, None, None, 0
for root, _dirs, files in os.walk(INPUT, followlinks=True):
    if "ledger.db" in files and ledger_path is None:
        ledger_path = os.path.join(root, "ledger.db")
    if "su2_cases.zip" in files and zip_path is None:
        zip_path = os.path.join(root, "su2_cases.zip")
    n = sum(1 for f in files if f.startswith("case_") and f.endswith(".npz"))
    if n > case_n:
        case_n, case_dir = n, root
print("ledger:", ledger_path)
print("zip:", zip_path)
print("cases:", case_n, "in", case_dir)
assert ledger_path is not None, f"no ledger.db under {INPUT}; attach the su2 dataset"

if case_n >= MIN_CASES:
    DATA = case_dir
    if not os.path.isfile(f"{DATA}/ledger.db"):
        DATA = "/kaggle/working/su2_data"
        os.makedirs(DATA, exist_ok=True)
        for f in os.listdir(case_dir):
            if f.startswith("case_") and f.endswith(".npz"):
                dst = f"{DATA}/{f}"
                if not os.path.exists(dst):
                    os.symlink(f"{case_dir}/{f}", dst)
        shutil.copyfile(ledger_path, f"{DATA}/ledger.db")
else:
    assert zip_path is not None, (
        f"only {case_n} case files and no su2_cases.zip under {INPUT}"
    )
    DATA = "/kaggle/working/su2_data"
    if not os.path.isfile(f"{DATA}/ledger.db"):
        os.makedirs(DATA, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(DATA)
        shutil.copyfile(ledger_path, f"{DATA}/ledger.db")

n_cases = len([d for d in os.listdir(DATA) if d.startswith("case_")])
print("DATA:", DATA, "cases:", n_cases)
assert n_cases >= MIN_CASES, (
    f"only {n_cases} case files under {DATA}, expected {MIN_CASES} or more"
)

M = 32


def run(cmd, log_dir=None):
    # the log also goes to the run directory, because cell output is lost on the next push
    lines = []
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    code = p.wait()
    if log_dir:
        os.makedirs(log_dir, exist_ok=True)
        with open(f"{log_dir}/train_log.txt", "w") as f:
            print("$", " ".join(cmd), file=f)
            print(file=f)
            f.writelines(lines)
    if code != 0:
        raise RuntimeError(f"run failed: {' '.join(cmd)}")

In [ ]:
RUN_ENSEMBLE = True
INIT_SEEDS = [2]
EPOCHS = 350

for s in INIT_SEEDS if RUN_ENSEMBLE else []:
    print(f"===== init_seed={s} =====", flush=True)
    OUT = f"/kaggle/working/run_m{M}_s{s}"
    run([sys.executable, "scripts/train.py",
         "--workdir", DATA,
         "--out", OUT,
         "--slice-num", str(M),
         "--epochs", str(EPOCHS),
         "--val-every", "10", "--seed", "0",
         "--init-seed", str(s)], log_dir=OUT)

## Physics-prior ablation

These cells price three priors the surrogate normally carries, one at a time, on
a training split of 100 cases. A prior earns its place when data is scarce, so
the small split is the design point and not a shortcut.

The evaluation tiers come from `configs/prior_splits_100.json`, so every cell is
scored on the same held-out cases at full size and only the training split is
small. Watch the split line. It must read `train: 100, val: 67, test: 65,
test_interp: 54`. Any other numbers mean the pinned file did not resolve against
the attached dataset, and the cell is void.

Every cell runs cosine decay over 500 epochs, unlike the 350-epoch recipe of
the ensemble, so these numbers compare only with each other. Put `base`,
`free_p` and `plain_norm` in one session and the two heat-flux cells in the
other, then repeat for seeds 1 and 2.

In [ ]:
RUN_PRIOR_ABLATION = False
PRIOR_SEED = 0
PRIOR_CELLS = ["base", "free_p", "plain_norm"]
PRIOR_EPOCHS = 500
PRIOR_SPLITS = f"{REPO}/configs/prior_splits_100.json"

PRIOR_FLAGS = {
    "base":       [],
    "free_p":     ["--predict-p"],
    "plain_norm": ["--log-targets", "none"],
    "qw_direct":  ["--qw-head", "direct"],
    "qw_resid":   ["--qw-head", "residual"],
}

for cell in PRIOR_CELLS if RUN_PRIOR_ABLATION else []:
    print(f"===== {cell} seed={PRIOR_SEED} =====", flush=True)
    OUT = f"/kaggle/working/run_{cell}_s{PRIOR_SEED}"
    run([sys.executable, "scripts/train.py",
         "--workdir", DATA,
         "--pinned-splits", PRIOR_SPLITS,
         "--out", OUT,
         "--slice-num", str(M),
         "--epochs", str(PRIOR_EPOCHS),
         "--lr-schedule", "cosine",
         "--val-every", "10", "--seed", "0",
         "--init-seed", str(PRIOR_SEED),
         *PRIOR_FLAGS[cell]], log_dir=OUT)

In [ ]:
import glob
import json

for path in sorted(glob.glob("/kaggle/working/run_*/final_eval.json")):
    with open(path) as f:
        final = json.load(f)
    a = final["args"]
    print("=====", path, "=====")
    print("  slice_num:", a["slice_num"], "| init_seed:", a.get("init_seed"),
          "| qw_head:", a.get("qw_head"), "| predict_p:", a.get("predict_p"))
    print("  best epoch:", final.get("best_epoch"), "of", a["epochs"],
          "| best val mean rL2:", final.get("best_val_mean_rL2"))
    print("  env:", final.get("env"))
    print(json.dumps(final["final"], indent=2))
    print("  splits:", final["splits"])